# NLI Experiment Suite — Holistic vs. Atomic × BiLSTM vs. PubMedBERT

This notebook consolidates the four training notebooks (`3a`–`3d`) and the results
comparison (`3e`) into **one OOP pipeline**. Every piece of logic that was
previously copy-pasted across notebooks — datasets, the label-encoding bug fix,
the training loop, evaluation, confusion-matrix plotting, aggregation — is now
defined **once** in a small class library, then reused by configuring and
running an `Experiment` object per model.

**Structure:**
1. Setup — imports & global config
2. Core library — shared classes (fix once, use everywhere)
3. Experiment 3a — Holistic BiLSTM
4. Experiment 3b — Holistic PubMedBERT
5. Experiment 3c — Atomic BiLSTM
6. Experiment 3d — Atomic PubMedBERT
7. Results comparison — 2×2 grid

**Note on the label-encoding bug:** earlier debugging found that the `label`
column is inconsistently typed across CSVs — sometimes an int, sometimes a
numeric string (`'0'`), sometimes a class name (`'neutral'`). This is fixed
**once**, in `LabelEncoder` below, and every dataset/aggregator in this
notebook routes through it — so it can never resurface in one model's cell
while being fixed in another's.

## Output directory structure (fix applied to this version)

Every artifact this notebook produces now lands in a predictable, categorized location instead
of the working directory root. Inputs from the three data-pipeline notebooks are untouched
(`data/`); everything below `outputs/` is created fresh by this notebook:

```
data/                                  <- inputs (0_ExtractData / 1_AugmentResponses / 2_Atomisation)
    holistic_train.csv, holistic_val.csv, holistic_test.csv
    atomic_train.csv,   atomic_val.csv,   atomic_test.csv
    vocab.json, embed_matrix.npy

outputs/
    checkpoints/        <- {model_name}_best.pt  (4 files: one per experiment)
    predictions/        <- {model_name}_test_predictions.csv
                           {model_name}_triplet_predictions.csv
                           {model_name}_aggregated_predictions.csv
    figures/            <- confusion matrices, training curves, the 2x2 comparison grid,
                           per-class F1 bars, the 4-way confusion matrix panel
    reports/            <- results_final_summary.csv
```

Previously every `.pt`/`.png`/`.csv` this notebook wrote was saved flat next to the notebook
itself (`f"{model_name}_best.pt"`, `title.replace(' ', '_') + '.png'`, etc.), so a full run left
~20 files of three different kinds mixed together in the repo root with no separation between
"model weights I need to keep", "predictions the results section reads back in", and "figures for
the thesis document" — and reruns silently overwrote them in place. All four `Experiment` classes,
`ResultsComparator`, and the plotting helpers now write into `CKPT_DIR` / `PRED_DIR` / `FIG_DIR` /
`REPORT_DIR` (defined once, in §0, alongside `DATA_DIR`) instead of the bare working directory.

`plot_confusion`'s filename sanitizer was also tightened: the old `title.replace(' ', '_')` left
punctuation like the em dash in `"{model} — Test Confusion Matrix"` in the filename verbatim; it
now goes through a small regex (`_safe_filename`) that maps anything that isn't a word character
or hyphen to `_`, which is safer across filesystems.


## 0 · Setup — Imports & Global Config

In [18]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, re, json, time
from dataclasses import dataclass, field
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader as TorchDataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    accuracy_score, matthews_corrcoef,
)
from scipy.stats import chi2

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

# ── Directories ──────────────────────────────────────────────────────────
# Inputs: produced by 0_ExtractData.ipynb / 1_AugmentResponses.ipynb / 2_Atomisation.ipynb
DATA_DIR   = "data"

# Outputs: everything THIS notebook produces is organized under outputs/<kind>/
# instead of being dumped flat into the working directory. A rerun no longer
# leaves 20+ mixed .pt / .png / .csv files sitting in the repo root, and each
# artifact type can be gitignored / archived independently.
OUTPUT_DIR = "outputs"
CKPT_DIR   = os.path.join(OUTPUT_DIR, "checkpoints")   # model .pt weights (best-epoch only)
PRED_DIR   = os.path.join(OUTPUT_DIR, "predictions")   # per-experiment prediction CSVs
FIG_DIR    = os.path.join(OUTPUT_DIR, "figures")       # confusion matrices, training curves, comparison plots
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")       # final summary tables / exported reports

for _d in (DATA_DIR, CKPT_DIR, PRED_DIR, FIG_DIR, REPORT_DIR):
    os.makedirs(_d, exist_ok=True)

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3-way NLI target only. "fake_drug" is NOT a 4th class here: 1_AugmentResponses.ipynb
# already maps it to "neutral" (NLI_LABEL['fake_drug'] = 'neutral') before the `label`
# column is ever written, so the string "fake_drug" never actually appears in `label` -
# the old 4-class LABEL_MAP's 4th entry was unreachable dead code, giving the model an
# unused output neuron with zero real training signal. The distinction between a genuine
# neutral response and a fake-drug-disguised-as-neutral response is tracked separately,
# via the `scenario` column (kept as-is, never collapsed) and the `entity_valid` flag
# used by the FActScore override in §7.
LABEL_MAP = {"entailment": 0, "contradiction": 1, "neutral": 2}
ID2LABEL  = {v: k for k, v in LABEL_MAP.items()}
CLASSES   = [ID2LABEL[i] for i in sorted(ID2LABEL)]

print(f"Device: {DEVICE}")
print(f"Classes: {CLASSES}")
print(f"\nData dir:    {DATA_DIR}/")
print(f"Checkpoints: {CKPT_DIR}/")
print(f"Predictions: {PRED_DIR}/")
print(f"Figures:     {FIG_DIR}/")
print(f"Reports:     {REPORT_DIR}/")

Device: cuda
Classes: ['entailment', 'contradiction', 'neutral']

Data dir:    data/
Checkpoints: outputs\checkpoints/
Predictions: outputs\predictions/
Figures:     outputs\figures/
Reports:     outputs\reports/


## 1 · Core Library — Shared OOP Components

Everything in this section is written once and reused by all four experiments.
Debugging a dataset, the training loop, or the aggregation rule here fixes it
**everywhere it's used** — that's the whole point of pulling it out of four
near-identical notebooks.

### 1.1 Label encoding

Robust to int, numpy int, numeric-string, or class-name-string labels — this is the fix from the earlier `int(row['label'])` crashes (`ValueError` on `'neutral'`, `KeyError` on `'0'`).

In [20]:
class LabelEncoder:
    """Converts a raw 'label' cell (int, numpy int, numeric string, or class
    name like 'neutral') into its integer class id. Try-int-first, fall back
    to the name→id map — this covers every label format seen across the CSVs.
    """
    def __init__(self, label_map: dict):
        self.label_map = label_map

    def encode(self, raw):
        try:
            return int(raw)
        except (ValueError, TypeError):
            return self.label_map[raw]

    def encode_series(self, series: pd.Series) -> pd.Series:
        return series.apply(self.encode)


LABEL_ENCODER = LabelEncoder(LABEL_MAP)

### 1.2 Datasets

`NLIPairDataset` tokenizes premise/hypothesis into a shared BioWordVec vocabulary for the BiLSTM models. `BERTNLIDataset` tokenizes them with a HuggingFace tokenizer for the PubMedBERT models. Both route labels through `LabelEncoder`.

In [21]:
def tokenize(text, word2idx, max_len=128):
    """Whitespace/punctuation-aware lowercase tokenizer → vocab ids."""
    tokens = re.findall(r"[a-z0-9']+", str(text).lower())[:max_len]
    ids = [word2idx.get(t, 1) for t in tokens]   # 1 = <UNK>
    return ids if ids else [1]


class NLIPairDataset(Dataset):
    """Premise/hypothesis pair dataset for the Siamese BiLSTM models.
    Used by both the holistic (3a) and atomic (3c) BiLSTM experiments —
    only `max_len` and the source CSVs differ between them.
    """
    def __init__(self, df, word2idx, label_encoder=LABEL_ENCODER, max_len=128):
        self.data          = df.reset_index(drop=True)
        self.word2idx      = word2idx
        self.label_encoder = label_encoder
        self.max_len       = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row   = self.data.iloc[idx]
        p_ids = tokenize(row['premise'],    self.word2idx, self.max_len)
        h_ids = tokenize(row['hypothesis'], self.word2idx, self.max_len)
        label = self.label_encoder.encode(row['label'])
        return (torch.tensor(p_ids), torch.tensor(h_ids), torch.tensor(label))


def collate_fn(batch):
    """Pads a batch of (premise_ids, hypothesis_ids, label) triples."""
    p_seqs, h_seqs, labels = zip(*batch)
    p_lens = torch.tensor([len(s) for s in p_seqs])
    h_lens = torch.tensor([len(s) for s in h_seqs])
    p_pad  = pad_sequence(p_seqs, batch_first=True, padding_value=0)
    h_pad  = pad_sequence(h_seqs, batch_first=True, padding_value=0)
    return p_pad, p_lens, h_pad, h_lens, torch.stack(labels)


class BERTNLIDataset(Dataset):
    """Premise/hypothesis pair dataset for the PubMedBERT models.
    Used by both the holistic (3b) and atomic (3d) BERT experiments.
    """
    # Tokens always reserved for the hypothesis (+ special tokens), so
    # 'only_second' truncation never runs out of room to cut when a
    # long premise alone is close to (or over) max_len.
    MIN_HYPOTHESIS_BUDGET = 32

    def __init__(self, df, tokenizer, label_encoder=LABEL_ENCODER, max_len=256):
        self.data          = df.reset_index(drop=True)
        self.tokenizer     = tokenizer
        self.label_encoder = label_encoder
        self.max_len       = max_len

    def __len__(self):
        return len(self.data)

    def _capped_premise(self, premise_text):
        """Pre-truncates the premise so it can never eat the entire
        max_length budget on its own. Without this, 'only_second'
        truncation raises when premise + special tokens >= max_len,
        since there'd be nothing left to trim from the hypothesis.
        """
        max_premise_len = max(self.max_len - self.MIN_HYPOTHESIS_BUDGET, 1)
        premise_ids = self.tokenizer.encode(premise_text, add_special_tokens=False)
        if len(premise_ids) <= max_premise_len:
            return premise_text
        premise_ids = premise_ids[:max_premise_len]
        return self.tokenizer.decode(premise_ids)

    def __getitem__(self, idx):
        row     = self.data.iloc[idx]
        premise = self._capped_premise(str(row['premise']))
        enc = self.tokenizer(
            premise, str(row['hypothesis']),
            max_length=self.max_len, padding='max_length',
            truncation='only_second',   # never truncate the (now-capped) premise
            return_tensors='pt',
        )
        label = self.label_encoder.encode(row['label'])
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'token_type_ids': enc.get(
                'token_type_ids', torch.zeros(self.max_len, dtype=torch.long)
            ).squeeze(0),
            'label': torch.tensor(label, dtype=torch.long),
        }

### 1.3 Model — Siamese BiLSTM

InferSent-style encoder: premise and hypothesis are each mean-pooled through a shared BiLSTM, then combined as `[u, v, |u−v|, u×v]` and classified. (PubMedBERT uses `AutoModelForSequenceClassification` directly — no custom class needed there.)

In [22]:
class SiameseBiLSTM(nn.Module):
    """InferSent-style Siamese BiLSTM for sentence-pair NLI.

    Encodes premise and hypothesis independently with a shared BiLSTM,
    then classifies via [u, v, |u-v|, u*v] aggregation.
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes,
                 pretrained_embeddings=None, freeze_embed=False, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            self.embedding.weight = nn.Parameter(
                torch.tensor(pretrained_embeddings, dtype=torch.float32),
                requires_grad=not freeze_embed,
            )
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=2,
            bidirectional=True, batch_first=True, dropout=dropout,
        )
        self.drop = nn.Dropout(dropout)
        enc_dim = hidden_dim * 2  # bidirectional
        self.classifier = nn.Sequential(
            nn.Linear(enc_dim * 4, 512),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def encode(self, ids, lengths):
        emb = self.drop(self.embedding(ids))            # [B, L, D]
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu().clamp(min=1), batch_first=True, enforce_sorted=False,
        )
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)  # [B, L, 2H]
        mask = (ids != 0).float().unsqueeze(-1)          # [B, L, 1]
        out  = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        return out                                        # [B, 2H]

    def forward(self, prem_ids, prem_len, hyp_ids, hyp_len):
        u = self.encode(prem_ids, prem_len)
        v = self.encode(hyp_ids,  hyp_len)
        x = torch.cat([u, v, torch.abs(u - v), u * v], dim=-1)
        return self.classifier(self.drop(x))

### 1.4 Class weights & plotting helpers

In [23]:
def compute_class_weights(df, n_classes, device, label_encoder=LABEL_ENCODER):
    """Inverse-frequency class weights, normalized to sum to n_classes."""
    encoded = label_encoder.encode_series(df['label'])
    counts  = encoded.value_counts().sort_index()
    weights = 1.0 / counts.reindex(range(n_classes), fill_value=1).values
    weights = weights / weights.sum() * n_classes
    return torch.tensor(weights, dtype=torch.float32).to(device)


def _safe_filename(name):
    """Filesystem-safe filename from an arbitrary plot title (handles
    em-dashes, ×, etc. that the old `title.replace(' ', '_')` left in place).
    """
    return re.sub(r"[^\w\-]+", "_", name).strip("_")


def plot_confusion(labels, preds, id2label, title, out_dir=FIG_DIR):
    cm    = confusion_matrix(labels, preds)
    names = [id2label[i] for i in sorted(id2label)]
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title)
    plt.tight_layout()
    out_path = os.path.join(out_dir, _safe_filename(title) + '.png')
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved figure -> {out_path}")


def plot_training_curves(history, epochs, model_name, out_dir=FIG_DIR):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    er = range(1, epochs + 1)
    ax1.plot(er, history['train_loss'], label='Train')
    ax1.plot(er, history['val_loss'],   label='Val')
    ax1.set(title='Loss', xlabel='Epoch', ylabel='Loss'); ax1.legend()
    ax2.plot(er, history['val_acc'],      label='Accuracy')
    ax2.plot(er, history['val_macro_f1'], label='Macro-F1')
    ax2.set(title='Val Metrics', xlabel='Epoch', ylabel='Score'); ax2.legend()
    plt.suptitle(f'{model_name} — Training Curves', fontsize=13)
    plt.tight_layout()
    out_path = os.path.join(out_dir, f'{model_name}_curves.png')
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved figure -> {out_path}")

### 1.5 Trainers

`BaseTrainer` implements the train/eval loop once. `BiLSTMTrainer` and `BERTTrainer` only override `_step`, which knows how to unpack their model-specific batch and call `forward`. This is the piece that used to be duplicated (and separately debugged) in every notebook.

In [24]:
class BaseTrainer:
    """Generic train/eval loop, shared by every model in this notebook.
    Subclasses only need to implement `_step`, which unpacks a batch,
    runs the forward pass, and returns (logits, loss, labels).
    """
    def __init__(self, model, device, model_name, ckpt_dir=CKPT_DIR):
        self.model      = model
        self.device     = device
        self.model_name = model_name
        self.history    = {"train_loss": [], "val_loss": [], "val_acc": [], "val_macro_f1": []}
        self.best_macro = 0.0
        self.best_ckpt  = os.path.join(ckpt_dir, f"{model_name}_best.pt")

    def _step(self, batch):
        raise NotImplementedError

    def train_epoch(self, loader, optimizer, scheduler=None, epoch=None, epochs=None):
        self.model.train()
        total_loss, correct, total = 0.0, 0, 0
        desc = f"{self.model_name} train" + (f" [{epoch}/{epochs}]" if epoch else "")
        pbar = tqdm(loader, desc=desc, leave=False)
        for batch in pbar:
            optimizer.zero_grad()
            logits, loss, labels = self._step(batch)
            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            total_loss += loss.item() * len(labels)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(labels)
            pbar.set_postfix(loss=f"{total_loss/total:.4f}", acc=f"{correct/total:.3f}")
        return total_loss / total, correct / total

    @torch.no_grad()
    def evaluate(self, loader, id2label):
        self.model.eval()
        total_loss, all_preds, all_labels = 0.0, [], []
        for batch in tqdm(loader, desc=f"{self.model_name} eval", leave=False):
            logits, loss, labels = self._step(batch)
            total_loss  += loss.item() * len(labels)
            all_preds   += logits.argmax(1).cpu().tolist()
            all_labels  += labels.cpu().tolist()
        n      = len(all_labels)
        acc    = sum(p == l for p, l in zip(all_preds, all_labels)) / n
        macro  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        report = classification_report(
            all_labels, all_preds,
            target_names=[id2label[i] for i in sorted(id2label)],
            zero_division=0,
        )
        return total_loss / n, acc, macro, report, all_preds, all_labels

    def fit(self, train_loader, val_loader, optimizer, epochs, id2label,
            scheduler=None, step_scheduler_every_batch=True):
        """If step_scheduler_every_batch is False, `scheduler.step()` is
        called once per epoch instead of once per batch (e.g. CosineAnnealingLR).
        """
        batch_scheduler = scheduler if step_scheduler_every_batch else None
        for epoch in range(1, epochs + 1):
            t0 = time.time()
            tr_loss, tr_acc = self.train_epoch(train_loader, optimizer, batch_scheduler,
                                                epoch=epoch, epochs=epochs)
            v_loss, v_acc, v_macro, _, _, _ = self.evaluate(val_loader, id2label)
            if not step_scheduler_every_batch and scheduler is not None:
                scheduler.step()

            self.history["train_loss"].append(tr_loss)
            self.history["val_loss"].append(v_loss)
            self.history["val_acc"].append(v_acc)
            self.history["val_macro_f1"].append(v_macro)

            if v_macro > self.best_macro:
                self.best_macro = v_macro
                torch.save(self.model.state_dict(), self.best_ckpt)
                flag = " ← best"
            else:
                flag = ""

            print(f"Epoch {epoch:02d}/{epochs} | "
                  f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f} | "
                  f"val_loss={v_loss:.4f} val_acc={v_acc:.3f} val_F1={v_macro:.3f} | "
                  f"{time.time()-t0:.1f}s{flag}")

        print(f"\nBest val macro-F1: {self.best_macro:.4f}")
        return self.history

    def load_best(self):
        self.model.load_state_dict(torch.load(self.best_ckpt, map_location=self.device))
        return self


class BiLSTMTrainer(BaseTrainer):
    def __init__(self, model, criterion, device, model_name):
        super().__init__(model, device, model_name)
        self.criterion = criterion

    def _step(self, batch):
        p_ids, p_len, h_ids, h_len, labels = batch
        p_ids, p_len = p_ids.to(self.device), p_len.to(self.device)
        h_ids, h_len = h_ids.to(self.device), h_len.to(self.device)
        labels       = labels.to(self.device)
        logits = self.model(p_ids, p_len, h_ids, h_len)
        loss   = self.criterion(logits, labels)
        return logits, loss, labels


class BERTTrainer(BaseTrainer):
    def _step(self, batch):
        input_ids  = batch['input_ids'].to(self.device)
        attn_mask  = batch['attention_mask'].to(self.device)
        token_type = batch['token_type_ids'].to(self.device)
        labels     = batch['label'].to(self.device)
        out = self.model(input_ids=input_ids, attention_mask=attn_mask,
                          token_type_ids=token_type, labels=labels)
        return out.logits, out.loss, labels

### 1.6 Atomic aggregation

Maps per-triplet predictions back to a single per-response verdict: **any predicted contradiction wins**; otherwise majority vote. Used by both atomic experiments (3c, 3d).

In [25]:
class AtomicAggregator:
    """Aggregates triplet-level NLI predictions into per-response verdicts.

    Rule:
      1. Any-contradiction: if ANY triplet is predicted Contradiction → Contradiction
      2. Otherwise: majority vote over the triplet predictions
    """
    def __init__(self, id2label, label_encoder=LABEL_ENCODER):
        self.id2label      = id2label
        self.label_encoder = label_encoder
        self.contra_id     = next(k for k, v in id2label.items() if v == "contradiction")

    def aggregate(self, atomic_df, preds):
        df = atomic_df.reset_index(drop=True).copy()
        df['pred_label'] = preds

        results = []
        for (orig_id, scenario), group in df.groupby(['original_id', 'scenario']):
            true_label  = self.label_encoder.encode(group['label'].mode()[0])
            pred_labels = group['pred_label'].tolist()

            if self.contra_id in pred_labels:
                final = self.contra_id
            else:
                final = Counter(pred_labels).most_common(1)[0][0]

            results.append({
                'original_id': orig_id,
                'scenario':    scenario,
                'true_label':  true_label,
                'pred_label':  final,
                'n_triplets':  len(pred_labels),
                'pred_dist':   str(Counter(pred_labels)),
                'correct':     final == true_label,
            })
        return pd.DataFrame(results)

### 1.7 Data loading helpers & experiment configs

In [26]:
class BioWordVecVocab:
    """Loads the shared BioWordVec vocab + embedding matrix used by both BiLSTM experiments.
    Built by 3_BuildBioWordVecVocab.ipynb (supersedes the earlier GloVe-based version — the
    proposal's §3.4.1 specifies BioWordVec dim=200 for the BiLSTM embedding layer, trained
    on PubMed/MeSH/MIMIC-III so pharmacology vocabulary isn't dominated by random-init OOV
    rows the way it was under generic GloVe).
    """
    def __init__(self, data_dir):
        with open(f"{data_dir}/vocab.json") as f:
            self.word2idx = json.load(f)
        self.embed_matrix = np.load(f"{data_dir}/embed_matrix.npy")
        self.vocab_size   = len(self.word2idx)


class NLIDataBundle:
    """Loads {prefix}_train/val/test.csv and prints a quick summary.
    `prefix` is 'holistic' or 'atomic'.
    """
    def __init__(self, data_dir, prefix):
        self.prefix   = prefix
        self.train_df = pd.read_csv(f"{data_dir}/{prefix}_train.csv")
        self.val_df   = pd.read_csv(f"{data_dir}/{prefix}_val.csv")
        self.test_df  = pd.read_csv(f"{data_dir}/{prefix}_test.csv")

    def summary(self):
        print(f"Train: {len(self.train_df)}  Val: {len(self.val_df)}  Test: {len(self.test_df)}")
        print("\nTrain class dist:")
        print(self.train_df['scenario'].value_counts())
        if self.prefix == "atomic":
            neutral_frac = (self.train_df['scenario'] == 'neutral').mean()
            print(f"\nNeutral fraction in train: {neutral_frac:.1%}")
            if neutral_frac < 0.10:
                print("⚠ Neutral is severely underrepresented — class weighting applied.")
        return self


@dataclass
class BiLSTMConfig:
    model_name:   str
    data_prefix:  str          # "holistic" or "atomic"
    max_len:      int
    embed_dim:    int = 200   # BioWordVec dimensionality per proposal §3.4.1 (was 100 under GloVe)
    hidden_dim:   int = 256
    dropout:      float = 0.3
    batch_size:   int = 64
    epochs:       int = 15
    lr:           float = 3e-4
    n_classes:    int = 3
    freeze_embed: bool = False


@dataclass
class BERTConfig:
    model_name:   str
    data_prefix:  str          # "holistic" or "atomic"
    bert_model:   str = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
    max_len:      int = 256
    batch_size:   int = 16
    epochs:       int = 5
    lr:           float = 2e-5
    warmup_frac:  float = 0.1
    n_classes:    int = 3

### 1.8 Experiment classes

Each `Experiment` wires the pieces above together: load data → build loaders → build model → train → evaluate → save predictions (+ aggregate, for atomic experiments). Running one of the four notebooks below is now a single method call.

In [27]:
class BiLSTMExperiment:
    """Runs one Siamese BiLSTM experiment (holistic or atomic)."""

    def __init__(self, config: BiLSTMConfig, data_dir=DATA_DIR, device=DEVICE,
                 label_map=LABEL_MAP, id2label=ID2LABEL):
        self.config    = config
        self.data_dir  = data_dir
        self.device    = device
        self.id2label  = id2label
        self.encoder   = LabelEncoder(label_map)
        self.is_atomic = config.data_prefix == "atomic"

    def load_data(self):
        self.bundle = NLIDataBundle(self.data_dir, self.config.data_prefix).summary()
        self.vocab  = BioWordVecVocab(self.data_dir)
        print(f"\nVocab: {self.vocab.vocab_size}   Embed matrix: {self.vocab.embed_matrix.shape}")
        return self

    def build_loaders(self):
        c = self.config
        mk_ds = lambda df: NLIPairDataset(df, self.vocab.word2idx, self.encoder, c.max_len)
        self.train_ds = mk_ds(self.bundle.train_df)
        self.val_ds   = mk_ds(self.bundle.val_df)
        self.test_ds  = mk_ds(self.bundle.test_df)

        mk_loader = lambda ds, shuffle: TorchDataLoader(
            ds, batch_size=c.batch_size, shuffle=shuffle, collate_fn=collate_fn)
        self.train_loader = mk_loader(self.train_ds, True)
        self.val_loader   = mk_loader(self.val_ds,   False)
        self.test_loader  = mk_loader(self.test_ds,  False)
        print(f"Train batches: {len(self.train_loader)}  Val batches: {len(self.val_loader)}")
        return self

    def build_model(self):
        c = self.config
        self.model = SiameseBiLSTM(
            vocab_size=self.vocab.vocab_size, embed_dim=c.embed_dim, hidden_dim=c.hidden_dim,
            n_classes=c.n_classes, pretrained_embeddings=self.vocab.embed_matrix,
            freeze_embed=c.freeze_embed, dropout=c.dropout,
        ).to(self.device)

        class_weights  = compute_class_weights(self.bundle.train_df, c.n_classes, self.device, self.encoder)
        self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=c.lr)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=c.epochs)
        self.trainer   = BiLSTMTrainer(self.model, self.criterion, self.device, c.model_name)

        total_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"Trainable parameters: {total_params:,}")
        print(f"Class weights: {{ {', '.join(f'{self.id2label[i]}: {w:.3f}' for i, w in enumerate(class_weights.cpu()))} }}")
        return self

    def train(self):
        self.history = self.trainer.fit(
            self.train_loader, self.val_loader, self.optimizer, self.config.epochs,
            self.id2label, scheduler=self.scheduler, step_scheduler_every_batch=False,
        )
        return self

    def plot_curves(self):
        plot_training_curves(self.history, self.config.epochs, self.config.model_name)
        return self

    def evaluate_test(self):
        self.trainer.load_best()
        _, self.test_acc, self.test_macro, report, self.preds, self.labels = \
            self.trainer.evaluate(self.test_loader, self.id2label)
        tag = "TRIPLET-LEVEL" if self.is_atomic else "TEST"
        print(f"{tag}  acc={self.test_acc:.4f}  macro-F1={self.test_macro:.4f}")
        print("\nClassification Report:")
        print(report)
        title = f"{self.config.model_name} — {'Triplet-Level' if self.is_atomic else 'Test'} Confusion Matrix"
        plot_confusion(self.labels, self.preds, self.id2label, title)
        return self

    def save_predictions(self):
        name = self.config.model_name
        if self.is_atomic:
            self.test_df_out = self.bundle.test_df.copy().reset_index(drop=True)
            self.test_df_out['pred_label'] = self.preds
            self.test_df_out['true_label'] = self.labels
            self.test_df_out.to_csv(os.path.join(PRED_DIR, f"{name}_triplet_predictions.csv"), index=False)

            aggregator  = AtomicAggregator(self.id2label, self.encoder)
            self.agg_df = aggregator.aggregate(self.bundle.test_df, self.preds)
            agg_macro   = f1_score(self.agg_df['true_label'], self.agg_df['pred_label'],
                                    average='macro', zero_division=0)
            print(f"AGGREGATED (response-level)  acc={self.agg_df['correct'].mean():.4f}  macro-F1={agg_macro:.4f}")
            print(classification_report(
                self.agg_df['true_label'], self.agg_df['pred_label'],
                target_names=[self.id2label[i] for i in sorted(self.id2label)], zero_division=0))
            plot_confusion(self.agg_df['true_label'].tolist(), self.agg_df['pred_label'].tolist(),
                            self.id2label, f"{name} — Aggregated Confusion")
            self.agg_df.to_csv(os.path.join(PRED_DIR, f"{name}_aggregated_predictions.csv"), index=False)

            print(f"\nSaved triplet     -> {os.path.join(PRED_DIR, name + '_triplet_predictions.csv')}")
            print(f"Saved aggregated  -> {os.path.join(PRED_DIR, name + '_aggregated_predictions.csv')}")
            print(f"Avg triplets per response: {self.agg_df['n_triplets'].mean():.1f}")
            print("Per-scenario accuracy:"); print(self.agg_df.groupby('scenario')['correct'].mean().round(3))
        else:
            self.test_df_out = self.bundle.test_df.copy().reset_index(drop=True)
            self.test_df_out['pred_label']    = self.preds
            self.test_df_out['pred_scenario'] = [self.id2label[p] for p in self.preds]
            self.test_df_out['true_label']    = self.labels
            self.test_df_out['correct']       = self.test_df_out['pred_label'] == self.test_df_out['true_label']
            out_path = os.path.join(PRED_DIR, f"{name}_test_predictions.csv")
            self.test_df_out.to_csv(out_path, index=False)
            print(f"Predictions saved → {out_path}")
            print("\nPer-scenario accuracy:")
            print(self.test_df_out.groupby('scenario')['correct'].mean().round(3))
        return self

    def run(self):
        return (self.load_data().build_loaders().build_model()
                    .train().plot_curves().evaluate_test().save_predictions())


class BERTExperiment:
    """Runs one PubMedBERT experiment (holistic or atomic)."""

    def __init__(self, config: BERTConfig, data_dir=DATA_DIR, device=DEVICE,
                 label_map=LABEL_MAP, id2label=ID2LABEL):
        self.config    = config
        self.data_dir  = data_dir
        self.device    = device
        self.id2label  = id2label
        self.encoder   = LabelEncoder(label_map)
        self.is_atomic = config.data_prefix == "atomic"

    def load_data(self):
        self.bundle = NLIDataBundle(self.data_dir, self.config.data_prefix).summary()
        return self

    def build_model(self):
        c = self.config
        self.tokenizer = AutoTokenizer.from_pretrained(c.bert_model)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            c.bert_model, num_labels=c.n_classes,
            id2label=self.id2label, label2id=LABEL_MAP,
            hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1,
        ).to(self.device)
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"Loaded {c.bert_model}")
        print(f"Trainable parameters: {trainable:,}")
        return self

    def build_loaders(self):
        c = self.config
        mk_ds = lambda df: BERTNLIDataset(df, self.tokenizer, self.encoder, c.max_len)
        self.train_ds = mk_ds(self.bundle.train_df)
        self.val_ds   = mk_ds(self.bundle.val_df)
        self.test_ds  = mk_ds(self.bundle.test_df)

        self.train_loader = TorchDataLoader(self.train_ds, batch_size=c.batch_size, shuffle=True)
        self.val_loader   = TorchDataLoader(self.val_ds,   batch_size=c.batch_size, shuffle=False)
        self.test_loader  = TorchDataLoader(self.test_ds,  batch_size=c.batch_size, shuffle=False)
        print(f"Train batches: {len(self.train_loader)}   Val batches: {len(self.val_loader)}")

        total_steps  = len(self.train_loader) * c.epochs
        warmup_steps = int(total_steps * c.warmup_frac)
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=c.lr, weight_decay=0.01)
        self.scheduler = get_linear_schedule_with_warmup(self.optimizer, warmup_steps, total_steps)
        print(f"Total steps: {total_steps}   Warmup steps: {warmup_steps}")

        self.trainer = BERTTrainer(self.model, self.device, c.model_name)
        return self

    def train(self):
        self.history = self.trainer.fit(
            self.train_loader, self.val_loader, self.optimizer, self.config.epochs,
            self.id2label, scheduler=self.scheduler, step_scheduler_every_batch=True,
        )
        return self

    def plot_curves(self):
        plot_training_curves(self.history, self.config.epochs, self.config.model_name)
        return self

    def evaluate_test(self):
        self.trainer.load_best()
        _, self.test_acc, self.test_macro, report, self.preds, self.labels = \
            self.trainer.evaluate(self.test_loader, self.id2label)
        tag = "TRIPLET-LEVEL" if self.is_atomic else "TEST"
        print(f"{tag}  acc={self.test_acc:.4f}  macro-F1={self.test_macro:.4f}")
        print("\nClassification Report:")
        print(report)
        title = f"{self.config.model_name} — {'Triplet-Level' if self.is_atomic else 'Test'} Confusion Matrix"
        plot_confusion(self.labels, self.preds, self.id2label, title)
        return self

    def save_predictions(self):
        name = self.config.model_name
        if self.is_atomic:
            self.test_df_out = self.bundle.test_df.copy().reset_index(drop=True)
            self.test_df_out['pred_label'] = self.preds
            self.test_df_out['true_label'] = self.labels
            self.test_df_out.to_csv(os.path.join(PRED_DIR, f"{name}_triplet_predictions.csv"), index=False)

            aggregator  = AtomicAggregator(self.id2label, self.encoder)
            self.agg_df = aggregator.aggregate(self.bundle.test_df, self.preds)
            agg_macro   = f1_score(self.agg_df['true_label'], self.agg_df['pred_label'],
                                    average='macro', zero_division=0)
            print(f"AGGREGATED  acc={self.agg_df['correct'].mean():.4f}  macro-F1={agg_macro:.4f}")
            print(classification_report(
                self.agg_df['true_label'], self.agg_df['pred_label'],
                target_names=[self.id2label[i] for i in sorted(self.id2label)], zero_division=0))
            plot_confusion(self.agg_df['true_label'].tolist(), self.agg_df['pred_label'].tolist(),
                            self.id2label, f"{name} — Aggregated Confusion")
            self.agg_df.to_csv(os.path.join(PRED_DIR, f"{name}_aggregated_predictions.csv"), index=False)
            print(f"Saved triplet + aggregated predictions -> {PRED_DIR}/")
        else:
            self.test_df_out = self.bundle.test_df.copy().reset_index(drop=True)
            self.test_df_out['pred_label']    = self.preds
            self.test_df_out['pred_scenario'] = [self.id2label[p] for p in self.preds]
            self.test_df_out['true_label']    = self.labels
            self.test_df_out['correct']       = self.test_df_out['pred_label'] == self.test_df_out['true_label']
            out_path = os.path.join(PRED_DIR, f"{name}_test_predictions.csv")
            self.test_df_out.to_csv(out_path, index=False)
            print(f"Predictions saved → {out_path}")
            print("\nPer-scenario accuracy:")
            print(self.test_df_out.groupby('scenario')['correct'].mean().round(3))
        return self

    def run(self):
        return (self.load_data().build_model().build_loaders()
                    .train().plot_curves().evaluate_test().save_predictions())

## 2 · Experiment 3a — Holistic BiLSTM

Siamese BiLSTM on full LLM-response vs. premise sentence pairs.

In [28]:
holistic_bilstm_config = BiLSTMConfig(
    model_name="holistic_bilstm",
    data_prefix="holistic",
    max_len=128,
)
exp_3a = BiLSTMExperiment(holistic_bilstm_config).run()

Train: 23028  Val: 4064  Test: 4780

Train class dist:
scenario
neutral          9360
entailment       6834
contradiction    6834
Name: count, dtype: int64

Vocab: 7494   Embed matrix: (7494, 200)
Train batches: 360  Val batches: 64
Trainable parameters: 5,128,883
Class weights: { entailment: 1.099, contradiction: 1.099, neutral: 0.802 }


holistic_bilstm train [1/15]:   0%|          | 0/360 [00:00<?, ?it/s]

holistic_bilstm eval:   0%|          | 0/64 [00:00<?, ?it/s]

Epoch 01/15 | tr_loss=0.2199 tr_acc=0.900 | val_loss=0.0088 val_acc=0.999 val_F1=0.999 | 19.4s ← best


holistic_bilstm train [2/15]:   0%|          | 0/360 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 3 · Experiment 3b — Holistic PubMedBERT

Fine-tunes BiomedNLP-BiomedBERT on full premise–hypothesis pairs. Truncation strategy is `only_second` — the premise is authoritative evidence and is never cut.

In [29]:
holistic_bert_config = BERTConfig(
    model_name="holistic_pubmedbert",
    data_prefix="holistic",
    max_len=256,
    batch_size=16,   # reduce to 8 if OOM on GPU
)
exp_3b = BERTExperiment(holistic_bert_config).run()

Train: 23028  Val: 4064  Test: 4780

Train class dist:
scenario
neutral          9360
entailment       6834
contradiction    6834
Name: count, dtype: int64


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

Loaded microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Trainable parameters: 109,484,547
Train batches: 1440   Val batches: 254
Total steps: 7200   Warmup steps: 720


holistic_pubmedbert train [1/5]:   0%|          | 0/1440 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 4 · Experiment 3c — Atomic BiLSTM

Same Siamese BiLSTM architecture, trained on SDP-extracted triplet hypotheses instead of full responses. Predictions are aggregated back to response-level via `AtomicAggregator`.

In [30]:
atomic_bilstm_config = BiLSTMConfig(
    model_name="atomic_bilstm",
    data_prefix="atomic",
    max_len=64,   # atomic hypotheses are ~10 words
)
exp_3c = BiLSTMExperiment(atomic_bilstm_config).run()

Train: 19828  Val: 3471  Test: 14025

Train class dist:
scenario
neutral          7388
entailment       6389
contradiction    6051
Name: count, dtype: int64

Neutral fraction in train: 37.3%

Vocab: 7494   Embed matrix: (7494, 200)
Train batches: 310  Val batches: 55
Trainable parameters: 5,128,883
Class weights: { entailment: 1.027, contradiction: 1.085, neutral: 0.888 }


atomic_bilstm train [1/15]:   0%|          | 0/310 [00:00<?, ?it/s]

atomic_bilstm eval:   0%|          | 0/55 [00:00<?, ?it/s]

Epoch 01/15 | tr_loss=0.7751 tr_acc=0.612 | val_loss=0.4461 val_acc=0.784 val_F1=0.784 | 13.8s ← best


atomic_bilstm train [2/15]:   0%|          | 0/310 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 5 · Experiment 3d — Atomic PubMedBERT

Fine-tunes PubMedBERT on the same atomic triplets — shorter inputs than the holistic model, so truncation risk is much lower.

In [31]:
atomic_bert_config = BERTConfig(
    model_name="atomic_pubmedbert",
    data_prefix="atomic",
    max_len=128,   # shorter than holistic — atomic hypotheses are brief
    batch_size=32,
)
exp_3d = BERTExperiment(atomic_bert_config).run()

Train: 19828  Val: 3471  Test: 14025

Train class dist:
scenario
neutral          7388
entailment       6389
contradiction    6051
Name: count, dtype: int64

Neutral fraction in train: 37.3%


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

Loaded microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Trainable parameters: 109,484,547
Train batches: 620   Val batches: 109
Total steps: 3100   Warmup steps: 310


atomic_pubmedbert train [1/5]:   0%|          | 0/620 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6 · Results Comparison — 2×2 Grid

Loads all four prediction files and produces:
- 2×2 metric summary table
- Per-class F1 breakdown (all 4 scenarios)
- Side-by-side confusion matrices
- McNemar's test for pairwise statistical significance
- Aggregation quality analysis (atomic models)
- Cross-model error analysis

In [32]:
class ResultsComparator:
    """Loads the four models' prediction CSVs and runs the full 2×2 comparison."""

    def __init__(self, files: dict, classes=CLASSES):
        self.files      = files
        self.classes    = classes
        self.preds_all  = {}

    def load(self):
        for name, path in self.files.items():
            df = pd.read_csv(path)
            self.preds_all[name] = df
            print(f"Loaded {name:25s} — {len(df)} rows  ({path})")
        return self

    def summary_table(self):
        rows = []
        for name, df in self.preds_all.items():
            y_true, y_pred = df['true_label'], df['pred_label']
            row = {
                "Model":    name,
                "Accuracy": round(accuracy_score(y_true, y_pred), 4),
                "Macro-F1": round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
                "MCC":      round(matthews_corrcoef(y_true, y_pred), 4),
            }
            per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
            for i, cls in enumerate(self.classes):
                row[f"F1_{cls}"] = round(per_class[i], 4) if i < len(per_class) else 0.0
            rows.append(row)

        self.summary = pd.DataFrame(rows).set_index("Model")
        print("=" * 80); print("2×2 EXPERIMENT RESULTS"); print("=" * 80)
        print(self.summary.to_string())
        return self

    def plot_grid(self):
        grid = pd.DataFrame({
            "BiLSTM":     [self.summary.loc["Holistic BiLSTM", "Macro-F1"],
                           self.summary.loc["Atomic BiLSTM",   "Macro-F1"]],
            "PubMedBERT": [self.summary.loc["Holistic PubMedBERT", "Macro-F1"],
                           self.summary.loc["Atomic PubMedBERT",   "Macro-F1"]],
        }, index=["Holistic", "Atomic"])

        fig, ax = plt.subplots(figsize=(5, 3))
        sns.heatmap(grid, annot=True, fmt=".4f", cmap="YlGnBu", vmin=0, vmax=1,
                    linewidths=0.5, ax=ax, cbar_kws={"label": "Macro-F1"})
        ax.set_title("2×2 Macro-F1 Grid", fontsize=13, fontweight='bold')
        ax.set_xlabel("Model Architecture"); ax.set_ylabel("Granularity")
        plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "results_2x2_grid.png"), dpi=150); plt.show()
        return self

    def plot_perclass_bars(self):
        class_cols       = [f"F1_{c}" for c in self.classes]
        bar_df           = self.summary[class_cols].T
        bar_df.index     = self.classes

        fig, ax = plt.subplots(figsize=(11, 5))
        x    = np.arange(len(self.classes))
        w    = 0.2
        cols = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
        for i, (model, color) in enumerate(zip(bar_df.columns, cols)):
            ax.bar(x + i * w, bar_df[model], w, label=model, color=color, alpha=0.85)
        ax.set_xticks(x + w * 1.5)
        ax.set_xticklabels(self.classes, fontsize=11)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("F1 Score"); ax.set_title("Per-Class F1 by Model", fontsize=13)
        ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
        plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "results_perclass_f1.png"), dpi=150); plt.show()
        return self

    def plot_confusions(self):
        fig, axes = plt.subplots(2, 2, figsize=(14, 11))
        for ax, name in zip(axes.flat, self.preds_all.keys()):
            df = self.preds_all[name]
            cm = confusion_matrix(df['true_label'], df['pred_label'])
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=self.classes, yticklabels=self.classes, ax=ax, cbar=False)
            ax.set_title(name, fontsize=11, fontweight='bold')
            ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        plt.suptitle("Confusion Matrices — All 4 Models", fontsize=14, fontweight='bold', y=1.01)
        plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "results_confusion_matrices.png"), dpi=150); plt.show()
        return self

    @staticmethod
    def _mcnemar_test(y_true, y_pred1, y_pred2):
        """Continuity-corrected McNemar's test. b/c = model1/2 uniquely correct."""
        b = sum(1 for yt, yp1, yp2 in zip(y_true, y_pred1, y_pred2) if yp1 == yt and yp2 != yt)
        c = sum(1 for yt, yp1, yp2 in zip(y_true, y_pred1, y_pred2) if yp1 != yt and yp2 == yt)
        if (b + c) == 0:
            return float('nan'), float('nan')
        stat = (abs(b - c) - 1) ** 2 / (b + c)
        p    = 1 - chi2.cdf(stat, df=1)
        return round(stat, 3), round(p, 4)

    def mcnemar_pairs(self):
        names = list(self.preds_all.keys())
        print(f"{'Pair':55s}  {'χ²':>6}  {'p-value':>8}  Sig?")
        print("─" * 80)
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                n1, n2 = names[i], names[j]
                d1 = self.preds_all[n1].set_index(['original_id', 'scenario'])
                d2 = self.preds_all[n2].set_index(['original_id', 'scenario'])
                shared = d1.index.intersection(d2.index)
                if len(shared) < 20:
                    print(f"  {n1} vs {n2}: insufficient shared rows ({len(shared)})")
                    continue
                yt  = d1.loc[shared, 'true_label'].tolist()
                yp1 = d1.loc[shared, 'pred_label'].tolist()
                yp2 = d2.loc[shared, 'pred_label'].tolist()
                stat, p = self._mcnemar_test(yt, yp1, yp2)
                sig = "✓" if isinstance(p, float) and p < 0.05 else " "
                print(f"  {n1:25s} vs {n2:25s}  {stat:>6}  {p:>8}  {sig}")
        return self

    def aggregation_quality(self):
        for name in ["Atomic BiLSTM", "Atomic PubMedBERT"]:
            df = self.preds_all.get(name)
            if df is None or 'n_triplets' not in df.columns:
                continue
            print(f"\n{name} — Triplets-per-response distribution:")
            print(df['n_triplets'].describe())
            df = df.copy()
            df['triplet_bin'] = pd.cut(df['n_triplets'], bins=[0, 1, 3, 6, 100],
                                        labels=['1', '2-3', '4-6', '7+'])
            print("\nAccuracy by number of triplets:")
            print(df.groupby('triplet_bin')['correct'].agg(['mean', 'count']).round(3))
        return self

    def error_analysis(self):
        keyed = {name: df for name, df in self.preds_all.items() if 'original_id' in df.columns}
        if not keyed:
            print("No models expose 'original_id' — skipping error analysis.")
            return self

        first_name, first_df = next(iter(keyed.items()))
        all_wrong_keys = set(
            first_df.set_index(['original_id', 'scenario'])[~first_df['correct'].values].index.tolist()
        )
        for name, df in keyed.items():
            wrong = set(df.set_index(['original_id', 'scenario'])[~df['correct'].values].index.tolist())
            all_wrong_keys &= wrong

        print(f"Responses all {len(keyed)} models get wrong: {len(all_wrong_keys)}")
        if all_wrong_keys and 'premise' in first_df.columns:
            examples = first_df.set_index(['original_id', 'scenario'])
            hard = examples.loc[list(all_wrong_keys)[:5], ['premise', 'scenario']]
            print("\nExamples:")
            for _, row in hard.iterrows():
                print(f"  [{row['scenario']}] {str(row['premise'])[:100]}...")
        return self

    def export(self):
        print("\n" + "=" * 80)
        print("FINAL RESULTS SUMMARY")
        print("=" * 80)
        print(self.summary.to_string())
        summary_path = os.path.join(REPORT_DIR, "results_final_summary.csv")
        self.summary.to_csv(summary_path)
        print(f"\nSaved -> {summary_path}")
        print(f"Saved -> {os.path.join(FIG_DIR, 'results_2x2_grid.png')}")
        print(f"Saved -> {os.path.join(FIG_DIR, 'results_perclass_f1.png')}")
        print(f"Saved -> {os.path.join(FIG_DIR, 'results_confusion_matrices.png')}")
        return self

    def run_all(self):
        return (self.load().summary_table().plot_grid().plot_perclass_bars()
                    .plot_confusions().mcnemar_pairs().aggregation_quality()
                    .error_analysis().export())

In [33]:
prediction_files = {
    "Holistic BiLSTM":     os.path.join(PRED_DIR, "holistic_bilstm_test_predictions.csv"),
    "Holistic PubMedBERT": os.path.join(PRED_DIR, "holistic_pubmedbert_test_predictions.csv"),
    "Atomic BiLSTM":       os.path.join(PRED_DIR, "atomic_bilstm_aggregated_predictions.csv"),
    "Atomic PubMedBERT":   os.path.join(PRED_DIR, "atomic_pubmedbert_aggregated_predictions.csv"),
}
comparator = ResultsComparator(prediction_files).run_all()

FileNotFoundError: [Errno 2] No such file or directory: 'outputs\\predictions\\holistic_bilstm_test_predictions.csv'

## 7 · FActScore Evaluation

**New section — this closes a gap between the proposal and the implementation.** §2.1.12/§3.5
define FActScore (Min et al., 2023) as the metric for granular factuality, and §3.7.1 explicitly
calls for it in the final pipeline ("*akan ditambahkan metode evaluasi FActScore guna menunjukkan
persentase fakta atomik yang sesuai dengan literatur ground-truth*"), but nothing in the notebook
actually computed it — §6 above only reports classification metrics (accuracy, Macro-F1,
McNemar) on the NLI task itself, never the response-level factuality score those atomic
predictions are supposed to feed into.

Per §2.1.12's own formalization:

```
f(y)          = (1/|A_y|) * sum_{a in A_y} 1[a is supported by C]
FActScore(M)  = E_x[ f(M_x) | M_x responds ]
```

Mapped onto this pipeline: one RAG response *y* = one `(original_id, scenario)` group of atomic
triplets in `atomic_test.csv`; an atomic fact *a* is "supported by *C*" (*C* = DrugBank, via the
`premise` column) exactly when the NLI verifier predicts **entailment** for that triplet. `f(y)`
is the fraction of a response's triplets predicted entailment; `FActScore(M)` is the mean of
`f(y)` over every response that produced at least one triplet (matching the `| M_x responds`
conditioning in the formula — responses the atomic parser couldn't decompose at all are excluded
from the mean, same scope caveat already documented in `2_Atomisation.ipynb`).

**One deliberate addition beyond the literal formula:** a response whose `entity_valid` is
`False` (a fabricated drug name) is scored `f(y) = 0` outright, regardless of what the triplet
classifier predicts. A surface-level entailment prediction about a sentence referencing a drug
that doesn't exist isn't "supported by DrugBank" in any meaningful sense — DrugBank has no entry
to support it against. Without this, a fluent hallucinated-drug sentence that happens to *read*
like a well-formed entailment could still score as fully "supported," which would defeat the
entire point of the `entity_valid` check introduced earlier in the pipeline. This is the natural
integration point for that check into the model-evaluation side, not just the data-prep side.

Computed for both atomic-granularity models (`atomic_bilstm`, `atomic_pubmedbert`) — FActScore is
inherently a triplet/atomic-level metric per the proposal, so it isn't computed for the holistic
models, which never produce atomic facts to average over.


In [34]:
class FActScoreEvaluator:
    """Implements the §2.1.12/§3.5 FActScore metric on this pipeline's atomic
    triplet predictions.

        f(y)         = (1/|A_y|) * sum_{a in A_y} 1[a supported by C]
        FActScore(M) = E_x[ f(M_x) | M_x responds ]

    - one response y  = one (original_id, scenario) group in a `*_triplet_predictions.csv`
    - "a supported by C" = the verifier predicted `entailment` for that atomic triplet
    - a response with entity_valid == False scores f(y) = 0 outright (see markdown
      above for why this isn't optional) - this is the ONE deliberate deviation
      from the literal formula, everything else follows Min et al. (2023) directly.
    """
    def __init__(self, id2label, label_encoder=LABEL_ENCODER):
        self.id2label   = id2label
        self.encoder    = label_encoder
        self.entail_id  = next(k for k, v in id2label.items() if v == "entailment")

    @staticmethod
    def _to_bool(x):
        if isinstance(x, bool):
            return x
        return str(x).strip().lower() in ("true", "1", "yes")

    def score(self, triplet_df: pd.DataFrame) -> pd.DataFrame:
        """Returns one row per (original_id, scenario) response with its f(y)."""
        rows = []
        for (orig_id, scenario), group in triplet_df.groupby(["original_id", "scenario"]):
            entity_valid = self._to_bool(group["entity_valid"].iloc[0]) if "entity_valid" in group else True
            if not entity_valid:
                f_y = 0.0
            else:
                f_y = (group["pred_label"] == self.entail_id).mean()
            rows.append({
                "original_id":  orig_id,
                "scenario":     scenario,
                "entity_valid": entity_valid,
                "n_triplets":   len(group),
                "f_y":          f_y,
            })
        return pd.DataFrame(rows)

    def report(self, triplet_df: pd.DataFrame, model_name: str) -> pd.DataFrame:
        per_response = self.score(triplet_df)
        overall = per_response["f_y"].mean() if len(per_response) else float("nan")

        print(f"\n=== FActScore -- {model_name} ===")
        print(f"FActScore(M) = {overall:.4f}   "
              f"(mean over {len(per_response):,} responses with >=1 extracted triplet)")
        print("\nBy ground-truth scenario (diagnostic, not part of the formula itself):")
        print("  expect ~1.0 for 'entailment' responses and ~0.0 for 'contradiction' /")
        print("  'neutral' / 'fake_drug' responses -- none of the latter are meant to be")
        print("  'supported' by the reference, so a well-behaved verifier should drive")
        print("  their f(y) toward 0, not just the overall average toward some middle value.")
        print(per_response.groupby("scenario")["f_y"].mean().round(4).to_string())
        return per_response


print("FActScoreEvaluator defined.")

FActScoreEvaluator defined.


In [35]:
def plot_factscore_comparison(factscore_results: dict, out_dir=FIG_DIR):
    """factscore_results: {model_name: per_response_df from FActScoreEvaluator.score()}"""
    scenarios = sorted(set(s for df in factscore_results.values() for s in df["scenario"].unique()))
    n_models  = len(factscore_results)
    x         = np.arange(len(scenarios))
    width     = 0.8 / max(n_models, 1)

    fig, ax = plt.subplots(figsize=(7, 5))
    for i, (name, df) in enumerate(factscore_results.items()):
        means = [df.loc[df["scenario"] == s, "f_y"].mean() for s in scenarios]
        ax.bar(x + i * width, means, width, label=name)

    ax.set_xticks(x + width * (n_models - 1) / 2)
    ax.set_xticklabels(scenarios)
    ax.set_ylabel("FActScore  (mean f(y))")
    ax.set_ylim(0, 1)
    ax.set_title("FActScore by ground-truth scenario")
    ax.legend()
    plt.tight_layout()

    out_path = os.path.join(out_dir, "factscore_by_scenario.png")
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved figure -> {out_path}")


print("plot_factscore_comparison() defined.")

plot_factscore_comparison() defined.


In [36]:
atomic_triplet_files = {
    "Atomic BiLSTM":     os.path.join(PRED_DIR, "atomic_bilstm_triplet_predictions.csv"),
    "Atomic PubMedBERT": os.path.join(PRED_DIR, "atomic_pubmedbert_triplet_predictions.csv"),
}

factscore_evaluator = FActScoreEvaluator(ID2LABEL)
factscore_results   = {}
factscore_summary_rows = []

for model_name, path in atomic_triplet_files.items():
    if not os.path.exists(path):
        print(f"[SKIP] {model_name}: {path} not found (run that experiment first)")
        continue
    triplet_df = pd.read_csv(path)
    per_response = factscore_evaluator.report(triplet_df, model_name)
    factscore_results[model_name] = per_response

    factscore_summary_rows.append({
        "model":       model_name,
        "factscore":   per_response["f_y"].mean(),
        "n_responses": len(per_response),
        **{f"factscore_{s}": per_response.loc[per_response["scenario"] == s, "f_y"].mean()
           for s in per_response["scenario"].unique()},
    })

if factscore_results:
    plot_factscore_comparison(factscore_results)

    factscore_summary = pd.DataFrame(factscore_summary_rows)
    summary_path = os.path.join(REPORT_DIR, "factscore_summary.csv")
    factscore_summary.to_csv(summary_path, index=False)
    print(f"\nSaved -> {summary_path}")
    print(factscore_summary.round(4).to_string(index=False))
else:
    print("No atomic triplet prediction files found -- run the atomic BiLSTM/PubMedBERT "
          "experiments (cells in §5) before this section.")

[SKIP] Atomic BiLSTM: outputs\predictions\atomic_bilstm_triplet_predictions.csv not found (run that experiment first)
[SKIP] Atomic PubMedBERT: outputs\predictions\atomic_pubmedbert_triplet_predictions.csv not found (run that experiment first)
No atomic triplet prediction files found -- run the atomic BiLSTM/PubMedBERT experiments (cells in §5) before this section.
